# ultraTM Data Pipeline — Binance Vision

**187 coins × 3 years (2023-04 ~ present)**

| Step | Data | Source | Est. Time |
|------|------|--------|-----------|
| 1 | OHLCV (1m, 5m, 15m, 1h) | monthly + daily klines | ~25min |
| 2 | Funding Rate | monthly fundingRate | ~5min |
| 3 | Metrics (OI, L/S ratio) | daily metrics | ~60min |
| 4 | aggTrades → tick_bar, volume_bar | monthly aggTrades | ~3-5hr |

Output: `Google Drive / ultraTM / data / merged / {SYMBOL} / *.parquet`

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q pyarrow requests tqdm psutil

In [ ]:
# ── Configuration ────────────────────────────────────────────────
OUTPUT_DIR       = "/content/drive/MyDrive/ultraTM/data/merged"
START_MONTH      = "2023-04"          # inclusive
INTERVALS        = ["1m", "5m", "15m", "1h"]
MIN_24H_VOL_USD  = 5_000_000          # $5M daily volume filter
METRICS_START    = "2023-04-01"        # metrics available from 2020-09
WORKERS          = 30                  # parallel download threads
TARGET_BARS_DAY  = 1000               # target bars/day for tick & volume bars

# Exclude stablecoins / index tokens
EXCLUDE_SYMBOLS  = {"USDCUSDT", "FDUSDUSDT", "DAIUSDT", "TUSDUSDT", "BTCDOMUSDT", "DEFIUSDT"}

# ── Multi-session sharding ──────────────────────────────────────
# Run multiple Colab tabs in parallel, each handling a slice of symbols.
# Set SHARD_ID (0-indexed) and TOTAL_SHARDS for each session.
# Example: 3 sessions → Tab1: 0/3, Tab2: 1/3, Tab3: 2/3
SHARD_ID      = 0    # which shard is THIS session (0, 1, 2, ...)
TOTAL_SHARDS  = 1    # total parallel sessions (1 = no sharding)


## 1. Core Utilities

In [ ]:
import gc
import io
import logging
import os
import time
import warnings
import zipfile
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import UTC, datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter, Retry
from tqdm.notebook import tqdm

# Suppress noisy warnings
logging.getLogger("urllib3.connectionpool").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

VISION = "https://data.binance.vision/data/futures/um"
FAPI   = "https://fapi.binance.com/fapi/v1"

KLINE_COLS   = ["open_time","open","high","low","close","volume",
                "close_time","quote_volume","count",
                "taker_buy_volume","taker_buy_quote_volume","ignore"]
FUNDING_COLS = ["calc_time","funding_interval_hours","last_funding_rate"]
METRICS_COLS = ["create_time","symbol","sum_open_interest","sum_open_interest_value",
                "count_toptrader_long_short_ratio","sum_toptrader_long_short_ratio",
                "count_long_short_ratio","sum_taker_long_short_vol_ratio"]
AGG_COLS     = ["agg_trade_id","price","quantity","first_trade_id",
                "last_trade_id","transact_time","is_buyer_maker"]

# Parallelism config
SYM_PARALLEL  = 8    # symbols at once (klines/funding/metrics)
URL_WORKERS   = 20   # download threads per symbol
AGG_PARALLEL  = 2    # aggTrades: fewer (memory-heavy)
MAX_CONN      = SYM_PARALLEL * URL_WORKERS + 50


def make_session(pool: int = MAX_CONN) -> requests.Session:
    s = requests.Session()
    retry = Retry(total=3, backoff_factor=0.5, status_forcelist=[500, 502, 503, 504])
    a = HTTPAdapter(pool_connections=pool, pool_maxsize=pool, max_retries=retry)
    s.mount("https://", a)
    return s


def months_between(start: str, end: str) -> list[str]:
    s = datetime.strptime(start, "%Y-%m")
    e = datetime.strptime(end, "%Y-%m")
    out = []
    while s <= e:
        out.append(s.strftime("%Y-%m"))
        s = (s.replace(day=28) + timedelta(days=4)).replace(day=1)
    return out


def dates_between(start: str, end: str) -> list[str]:
    s = datetime.strptime(start, "%Y-%m-%d")
    e = datetime.strptime(end, "%Y-%m-%d")
    out = []
    while s <= e:
        out.append(s.strftime("%Y-%m-%d"))
        s += timedelta(days=1)
    return out


def last_complete_month() -> str:
    t = datetime.now(UTC)
    return (t.replace(day=1) - timedelta(days=1)).strftime("%Y-%m")


def current_month_dates() -> list[str]:
    t = datetime.now(UTC)
    first = t.replace(day=1)
    yest = t - timedelta(days=1)
    if yest < first:
        return []
    return dates_between(first.strftime("%Y-%m-%d"), yest.strftime("%Y-%m-%d"))


def fetch_zip_csv(url: str, session: requests.Session) -> pd.DataFrame | None:
    try:
        r = session.get(url, timeout=30)
        if r.status_code == 404:
            return None
        r.raise_for_status()
    except requests.RequestException:
        return None
    try:
        with zipfile.ZipFile(io.BytesIO(r.content)) as zf:
            with zf.open(zf.namelist()[0]) as f:
                df = pd.read_csv(f, header=None, dtype=str, low_memory=False)
    except Exception:
        return None
    if df.empty:
        return None
    # Drop header row if present
    try:
        float(df.iloc[0, 0].strip())
    except (ValueError, AttributeError):
        df = df.iloc[1:].reset_index(drop=True)
    return df


def fetch_many(urls: list[str], session: requests.Session,
               workers: int = URL_WORKERS) -> list[pd.DataFrame]:
    results = []
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futs = {pool.submit(fetch_zip_csv, u, session): u for u in urls}
        for fut in as_completed(futs):
            df = fut.result()
            if df is not None and not df.empty:
                results.append(df)
    return results


def save_parquet(df: pd.DataFrame, path: str):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False, engine="pyarrow")


SESSION = make_session()
MONTHS  = months_between(START_MONTH, last_complete_month())
DAILY   = current_month_dates()
YESTERDAY = (datetime.now(UTC) - timedelta(days=1)).strftime("%Y-%m-%d")
METRIC_DATES = dates_between(METRICS_START, YESTERDAY)

print(f"Monthly range : {MONTHS[0]} ~ {MONTHS[-1]} ({len(MONTHS)} months)")
print(f"Daily  (cur)  : {len(DAILY)} days")
print(f"Metrics range : {METRICS_START} ~ {YESTERDAY} ({len(METRIC_DATES)} days)")
print(f"Parallelism   : {SYM_PARALLEL} sym x {URL_WORKERS} workers | aggTrades: {AGG_PARALLEL} sym")


## 2. Symbol Discovery

In [ ]:
# Hardcoded fallback — Binance Futures API is blocked on some cloud IPs (Colab/GCP)
# Generated 2026-04-06, vol >= $5M, excluding stablecoins/index/chinese meme tokens
FALLBACK_SYMBOLS = [
    "0GUSDT", "1000BONKUSDT", "1000FLOKIUSDT", "1000LUNCUSDT", "1000PEPEUSDT",
    "1000SHIBUSDT", "1INCHUSDT", "4USDT", "AAVEUSDT", "ADAUSDT", "AEVOUSDT",
    "AIAUSDT", "AIOTUSDT", "ALGOUSDT", "ANKRUSDT", "APEUSDT", "APTUSDT", "ARBUSDT",
    "ARIAUSDT", "ASTERUSDT", "ATOMUSDT", "AVAXUSDT", "AXSUSDT", "BABYUSDT",
    "BANANAS31USDT", "BANKUSDT", "BARDUSDT", "BASEDUSDT", "BASUSDT", "BBUSDT",
    "BCHUSDT", "BEATUSDT", "BERAUSDT", "BIOUSDT", "BLURUSDT", "BNBUSDT",
    "BROCCOLI714USDT", "BROCCOLIF3BUSDT", "BRUSDT", "BSBUSDT", "BSVUSDT",
    "BTCUSDT", "BULLAUSDT", "CAKEUSDT", "CELOUSDT", "CFGUSDT", "CHRUSDT",
    "CHZUSDT", "COLLECTUSDT", "COSUSDT", "CRVUSDT", "CTSIUSDT", "CUSDT",
    "CYSUSDT", "DASHUSDT", "DEGOUSDT", "DEXEUSDT", "DOGEUSDT", "DOLOUSDT",
    "DOTUSDT", "DRIFTUSDT", "DUSDT", "DYDXUSDT", "EDGEUSDT", "EDUUSDT",
    "EIGENUSDT", "ENAUSDT", "ENSOUSDT", "ESPORTSUSDT", "ETCUSDT", "ETHFIUSDT",
    "ETHUSDT", "FARTCOINUSDT", "FETUSDT", "FIDAUSDT", "FILUSDT", "FOGOUSDT",
    "GALAUSDT", "GASUSDT", "GIGGLEUSDT", "GUAUSDT", "GUSDT", "HBARUSDT",
    "HEMIUSDT", "HIPPOUSDT", "HUMAUSDT", "HUSDT", "HYPEUSDT", "ICPUSDT",
    "INJUSDT", "IPUSDT", "JCTUSDT", "JTOUSDT", "KATUSDT", "KERNELUSDT",
    "KITEUSDT", "KOMAUSDT", "LDOUSDT", "LINEAUSDT", "LINKUSDT", "LITUSDT",
    "LTCUSDT", "LYNUSDT", "MASKUSDT", "MEUSDT", "MMTUSDT", "MONUSDT", "MUSDT",
    "MYXUSDT", "NEARUSDT", "NIGHTUSDT", "NMRUSDT", "NOMUSDT", "ONDOUSDT",
    "ONGUSDT", "ONTUSDT", "ONUSDT", "OPNUSDT", "OPUSDT", "ORDIUSDT", "PAXGUSDT",
    "PENDLEUSDT", "PENGUUSDT", "PIPPINUSDT", "PIXELUSDT", "PLAYUSDT", "POLUSDT",
    "POLYXUSDT", "POWERUSDT", "PRLUSDT", "PTBUSDT", "PUFFERUSDT", "PUMPUSDT",
    "QNTUSDT", "RENDERUSDT", "RESOLVUSDT", "REZUSDT", "RIVERUSDT", "RLSUSDT",
    "ROBOUSDT", "SAFEUSDT", "SAGAUSDT", "SAHARAUSDT", "SANDUSDT", "SEIUSDT",
    "SIGNUSDT", "SIRENUSDT", "SKYAIUSDT", "SNXUSDT", "SOLUSDT", "SOLVUSDT",
    "STABLEUSDT", "STGUSDT", "STOUSDT", "SUIUSDT", "SWARMSUSDT", "TAOUSDT",
    "THETAUSDT", "THEUSDT", "TIAUSDT", "TONUSDT", "TRADOORUSDT", "TRIAUSDT",
    "TRUMPUSDT", "TRUSTUSDT", "TRXUSDT", "TUSDT", "UNIUSDT", "USTCUSDT",
    "VETUSDT", "VIRTUALUSDT", "VVVUSDT", "WIFUSDT", "WLDUSDT", "WLFIUSDT",
    "WUSDT", "XAUTUSDT", "XLMUSDT", "XMRUSDT", "XPLUSDT", "XRPUSDT", "YBUSDT",
    "ZBTUSDT", "ZECUSDT", "ZENUSDT", "ZETAUSDT", "ZKUSDT", "ZROUSDT",
]

# Additional exclusions beyond stablecoins
EXCLUDE_SYMBOLS.update({
    # Chinese meme tokens
    "币安人生USDT", "我踏马来了USDT", "龙虾USDT",
    # Gold/commodity tokens (not crypto)
    "XAUTUSDT", "PAXGUSDT", "XAUUSDT", "XAGUSDT",
})

# Try live API first, fall back to hardcoded list
try:
    info = requests.get(f"{FAPI}/exchangeInfo", timeout=15).json()
    all_perps = sorted([
        s["symbol"] for s in info["symbols"]
        if s["contractType"] == "PERPETUAL"
        and s["quoteAsset"] == "USDT"
        and s["status"] == "TRADING"
    ])
    print(f"API OK — {len(all_perps)} active perpetuals")

    tickers = {t["symbol"]: float(t["quoteVolume"])
               for t in requests.get(f"{FAPI}/ticker/24hr", timeout=15).json()}
    symbols = sorted([
        s for s in all_perps
        if s not in EXCLUDE_SYMBOLS
        and tickers.get(s, 0) >= MIN_24H_VOL_USD
    ])
    print(f"After filter (vol >= ${MIN_24H_VOL_USD/1e6:.0f}M): {len(symbols)}")

except Exception as e:
    print(f"Binance API blocked ({e}), using fallback list")
    symbols = [s for s in FALLBACK_SYMBOLS if s not in EXCLUDE_SYMBOLS]
    print(f"Fallback symbols: {len(symbols)}")

# Save symbol list
sym_path = f"{OUTPUT_DIR}/symbols.txt"
Path(sym_path).parent.mkdir(parents=True, exist_ok=True)
Path(sym_path).write_text("\n".join(symbols) + "\n")
print(f"Saved to {sym_path}")

# Apply sharding
if TOTAL_SHARDS > 1:
    symbols = [s for i, s in enumerate(symbols) if i % TOTAL_SHARDS == SHARD_ID]
    print(f"Shard {SHARD_ID}/{TOTAL_SHARDS}: {len(symbols)} symbols for THIS session")


## 3. Download Klines (1m, 5m, 15m, 1h)

In [ ]:
def _process_kline_dfs(dfs):
    """Concat raw kline DataFrames → clean parquet-ready DataFrame."""
    df = pd.concat(dfs, ignore_index=True)
    df.columns = KLINE_COLS[:df.shape[1]]
    for c in ["open","high","low","close","volume","quote_volume",
              "taker_buy_volume","taker_buy_quote_volume"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    out = pd.DataFrame()
    out["timestamp"] = pd.to_datetime(pd.to_numeric(df["open_time"], errors="coerce"), unit="ms", utc=True)
    for c in ["open","high","low","close","volume","quote_volume"]:
        out[c] = df[c].values
    if "count" in df.columns:
        out["trade_count"] = pd.to_numeric(df["count"], errors="coerce").astype("Int64").values
    if "taker_buy_volume" in df.columns:
        out["taker_buy_volume"] = df["taker_buy_volume"].values
    if "taker_buy_quote_volume" in df.columns:
        out["taker_buy_quote_volume"] = df["taker_buy_quote_volume"].values
    return out.drop_duplicates("timestamp").sort_values("timestamp").reset_index(drop=True)


def download_save_kline(sym_interval):
    """Download + save one (symbol, interval). Returns (sym, interval, status, rows)."""
    sym, interval = sym_interval
    path = f"{OUTPUT_DIR}/{sym}/kline_{interval}.parquet"
    if os.path.exists(path):
        return sym, interval, "skip", 0
    urls = (
        [f"{VISION}/monthly/klines/{sym}/{interval}/{sym}-{interval}-{m}.zip" for m in MONTHS]
        + [f"{VISION}/daily/klines/{sym}/{interval}/{sym}-{interval}-{d}.zip" for d in DAILY]
    )
    dfs = fetch_many(urls, SESSION)
    if not dfs:
        return sym, interval, "no_data", 0
    out = _process_kline_dfs(dfs)
    save_parquet(out, path)
    n = len(out)
    del out, dfs
    return sym, interval, "ok", n


# Build task list — skip existing files
pairs = [(sym, iv) for sym in symbols for iv in INTERVALS
         if not os.path.exists(f"{OUTPUT_DIR}/{sym}/kline_{iv}.parquet")]

already = len(symbols) * len(INTERVALS) - len(pairs)
print(f"Klines: {len(pairs)} to download, {already} already exist (skipped)")
print(f"Parallel: {SYM_PARALLEL} × {URL_WORKERS} = {SYM_PARALLEL * URL_WORKERS} concurrent downloads\n")

ok, fail = 0, 0
with ThreadPoolExecutor(max_workers=SYM_PARALLEL) as pool:
    futs = {pool.submit(download_save_kline, p): p for p in pairs}
    for fut in tqdm(as_completed(futs), total=len(futs), desc="Klines"):
        sym, interval, status, n = fut.result()
        if status == "ok":
            ok += 1
            tqdm.write(f"  {sym} {interval}: {n:,} rows")
        elif status == "no_data":
            fail += 1

print(f"\nDone. saved={ok}, no_data={fail}, skipped={already}")

## 4. Download Funding Rate

In [ ]:
def download_save_funding(sym):
    """Download + save funding rate for one symbol."""
    path = f"{OUTPUT_DIR}/{sym}/funding_rate.parquet"
    if os.path.exists(path):
        return sym, "skip", 0
    urls = [f"{VISION}/monthly/fundingRate/{sym}/{sym}-fundingRate-{m}.zip" for m in MONTHS]
    dfs = fetch_many(urls, SESSION)
    if not dfs:
        return sym, "no_data", 0
    df = pd.concat(dfs, ignore_index=True)
    df.columns = FUNDING_COLS[:df.shape[1]]
    out = pd.DataFrame()
    out["timestamp"] = pd.to_datetime(pd.to_numeric(df["calc_time"], errors="coerce"), unit="ms", utc=True)
    out["funding_interval_hours"] = pd.to_numeric(df["funding_interval_hours"], errors="coerce").astype("Int64")
    out["funding_rate"] = pd.to_numeric(df["last_funding_rate"], errors="coerce")
    out = out.drop_duplicates("timestamp").sort_values("timestamp").reset_index(drop=True)
    save_parquet(out, path)
    n = len(out)
    del out, dfs
    return sym, "ok", n


todo = [s for s in symbols if not os.path.exists(f"{OUTPUT_DIR}/{s}/funding_rate.parquet")]
already = len(symbols) - len(todo)
print(f"Funding: {len(todo)} to download, {already} already exist (skipped)\n")

ok, fail = 0, 0
with ThreadPoolExecutor(max_workers=SYM_PARALLEL * 2) as pool:
    futs = {pool.submit(download_save_funding, s): s for s in todo}
    for fut in tqdm(as_completed(futs), total=len(futs), desc="Funding"):
        sym, status, n = fut.result()
        if status == "ok":
            ok += 1
        elif status == "no_data":
            fail += 1

print(f"\nDone. saved={ok}, no_data={fail}, skipped={already}")

## 5. Download Metrics

In [ ]:
def download_save_metrics(sym):
    """Download + save metrics for one symbol."""
    path = f"{OUTPUT_DIR}/{sym}/metrics.parquet"
    if os.path.exists(path):
        return sym, "skip", 0
    urls = [f"{VISION}/daily/metrics/{sym}/{sym}-metrics-{d}.zip" for d in METRIC_DATES]
    dfs = fetch_many(urls, SESSION, workers=30)  # more workers for many daily files
    if not dfs:
        return sym, "no_data", 0
    df = pd.concat(dfs, ignore_index=True)
    df.columns = METRICS_COLS[:df.shape[1]]
    out = pd.DataFrame()
    out["timestamp"] = pd.to_datetime(df["create_time"], errors="coerce", utc=True)
    out["symbol"] = sym
    for c in ["sum_open_interest","sum_open_interest_value",
              "count_toptrader_long_short_ratio","sum_toptrader_long_short_ratio",
              "count_long_short_ratio","sum_taker_long_short_vol_ratio"]:
        if c in df.columns:
            out[c] = pd.to_numeric(df[c], errors="coerce").values
    out = out.drop_duplicates("timestamp").sort_values("timestamp").reset_index(drop=True)
    save_parquet(out, path)
    n = len(out)
    del out, dfs
    return sym, "ok", n


todo = [s for s in symbols if not os.path.exists(f"{OUTPUT_DIR}/{s}/metrics.parquet")]
already = len(symbols) - len(todo)
print(f"Metrics: {len(todo)} to download, {already} already exist (skipped)\n")

ok, fail = 0, 0
with ThreadPoolExecutor(max_workers=SYM_PARALLEL) as pool:
    futs = {pool.submit(download_save_metrics, s): s for s in todo}
    for fut in tqdm(as_completed(futs), total=len(futs), desc="Metrics"):
        sym, status, n = fut.result()
        if status == "ok":
            ok += 1
            tqdm.write(f"  {sym}: {n:,} rows")
        elif status == "no_data":
            fail += 1

print(f"\nDone. saved={ok}, no_data={fail}, skipped={already}")

## 6. aggTrades → tick_bar + volume_bar

Downloads aggTrades **one month at a time**, converts to bars in memory, then discards raw data.

Bar size is **calibrated per symbol** to target ~1000 bars/day.

In [ ]:
def fetch_agg_daily(sym: str, date: str, session: requests.Session) -> pd.DataFrame | None:
    url = f"{VISION}/daily/aggTrades/{sym}/{sym}-aggTrades-{date}.zip"
    try:
        r = session.get(url, timeout=60)
        if r.status_code == 404:
            return None
        r.raise_for_status()
    except requests.RequestException:
        return None
    try:
        with zipfile.ZipFile(io.BytesIO(r.content)) as zf:
            with zf.open(zf.namelist()[0]) as f:
                df = pd.read_csv(f, header=None, dtype=str, low_memory=False)
    except Exception:
        return None
    if df.empty:
        return None
    try:
        float(df.iloc[0, 0].strip())
    except (ValueError, AttributeError):
        df = df.iloc[1:].reset_index(drop=True)
    df.columns = AGG_COLS[:df.shape[1]]
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
    df["transact_time"] = pd.to_numeric(df["transact_time"], errors="coerce")
    df["is_buyer_maker"] = df["is_buyer_maker"].astype(str).str.lower().isin(["true", "1"])
    return df[["price", "quantity", "transact_time", "is_buyer_maker"]]


def build_tick_bars(trades, bar_size):
    trades = trades.sort_values("transact_time").reset_index(drop=True)
    trades["bar_id"] = np.arange(len(trades)) // bar_size
    buy_mask = ~trades["is_buyer_maker"]
    trades["buy_qty"] = trades["quantity"].where(buy_mask, 0)
    trades["sell_qty"] = trades["quantity"].where(~buy_mask, 0)
    return trades.groupby("bar_id").agg(
        timestamp=("transact_time", "first"),
        open=("price", "first"), high=("price", "max"),
        low=("price", "min"), close=("price", "last"),
        volume=("quantity", "sum"), trade_count=("price", "count"),
        buy_volume=("buy_qty", "sum"), sell_volume=("sell_qty", "sum"),
    ).reset_index(drop=True).assign(
        timestamp=lambda x: pd.to_datetime(x["timestamp"], unit="ms", utc=True)
    )


def build_volume_bars(trades, vol_threshold):
    trades = trades.sort_values("transact_time").reset_index(drop=True)
    trades["bar_id"] = (trades["quantity"].cumsum() / vol_threshold).astype(int)
    buy_mask = ~trades["is_buyer_maker"]
    trades["buy_qty"] = trades["quantity"].where(buy_mask, 0)
    trades["sell_qty"] = trades["quantity"].where(~buy_mask, 0)
    return trades.groupby("bar_id").agg(
        timestamp=("transact_time", "first"),
        open=("price", "first"), high=("price", "max"),
        low=("price", "min"), close=("price", "last"),
        volume=("quantity", "sum"), trade_count=("price", "count"),
        buy_volume=("buy_qty", "sum"), sell_volume=("sell_qty", "sum"),
    ).reset_index(drop=True).assign(
        timestamp=lambda x: pd.to_datetime(x["timestamp"], unit="ms", utc=True)
    )


def get_symbol_start_date(sym):
    """Get actual start date from kline_1m data."""
    p = f"{OUTPUT_DIR}/{sym}/kline_1m.parquet"
    if os.path.exists(p):
        df = pd.read_parquet(p, columns=["timestamp"])
        return df["timestamp"].min().strftime("%Y-%m-%d")
    return AGG_DATES[0]


def calibrate_bar_sizes(trades_sample, n_days):
    daily_trades = len(trades_sample) / max(n_days, 1)
    daily_volume = trades_sample["quantity"].sum() / max(n_days, 1)
    tick_size = max(int(daily_trades / TARGET_BARS_DAY), 1)
    vol_size = max(daily_volume / TARGET_BARS_DAY, trades_sample["quantity"].min())
    return tick_size, vol_size


BATCH_DAYS = 30

def download_save_aggtrades(sym):
    tick_path = f"{OUTPUT_DIR}/{sym}/tick_bar.parquet"
    vol_path  = f"{OUTPUT_DIR}/{sym}/volume_bar.parquet"
    if os.path.exists(tick_path) and os.path.exists(vol_path):
        return sym, "skip", 0, 0

    # Use coin's actual start date, not global start
    start = get_symbol_start_date(sym)
    sym_dates = [d for d in AGG_DATES if d >= start]
    if not sym_dates:
        return sym, "no_dates", 0, 0

    # Calibrate from first available days OF THIS COIN
    cal_dfs = []
    for d in sym_dates[:14]:  # try up to 14 days
        df = fetch_agg_daily(sym, d, SESSION)
        if df is not None and len(df) > 100:
            cal_dfs.append(df)
        if len(cal_dfs) >= 3:
            break
    if not cal_dfs:
        return sym, "no_data", 0, 0

    sample = pd.concat(cal_dfs, ignore_index=True)
    tick_size, vol_size = calibrate_bar_sizes(sample, len(cal_dfs))
    del sample, cal_dfs

    # Process in batches
    all_tick, all_vol = [], []
    for i in range(0, len(sym_dates), BATCH_DAYS):
        batch = sym_dates[i:i+BATCH_DAYS]
        batch_dfs = {}
        with ThreadPoolExecutor(max_workers=15) as pool:
            futs = {pool.submit(fetch_agg_daily, sym, d, SESSION): d for d in batch}
            for fut in as_completed(futs):
                d = futs[fut]
                r = fut.result()
                if r is not None and len(r) >= 10:
                    batch_dfs[d] = r
        for d in sorted(batch_dfs.keys()):
            all_tick.append(build_tick_bars(batch_dfs[d], tick_size))
            all_vol.append(build_volume_bars(batch_dfs[d], vol_size))
        del batch_dfs

    nt, nv = 0, 0
    if all_tick:
        df = pd.concat(all_tick, ignore_index=True)
        save_parquet(df, tick_path)
        nt = len(df)
        del df
    del all_tick
    if all_vol:
        df = pd.concat(all_vol, ignore_index=True)
        save_parquet(df, vol_path)
        nv = len(df)
        del df
    del all_vol
    gc.collect()
    return sym, "ok", nt, nv


AGG_DATES = dates_between(MONTHS[0] + "-01", YESTERDAY)

print(f"aggTrades range: {AGG_DATES[0]} ~ {AGG_DATES[-1]} ({len(AGG_DATES)} days)")
print("Calibration: uses each coin's actual listing date (from kline_1m)")


In [ ]:
import psutil

BATCH = 30       # days per parallel batch
DL_WORKERS = 15  # parallel downloads per batch

todo = [s for s in symbols
        if not (os.path.exists(f"{OUTPUT_DIR}/{s}/tick_bar.parquet")
                and os.path.exists(f"{OUTPUT_DIR}/{s}/volume_bar.parquet"))]
already = len(symbols) - len(todo)
print(f"aggTrades: {len(todo)} to process, {already} skipped")
print(f"RAM: {psutil.virtual_memory().available/1e9:.1f}GB free\n")

for si, sym in enumerate(todo):
    print(f"━━━ [{si+1}/{len(todo)}] {sym} ━━━")

    # 1. Calibrate
    print(f"  Calibrating...", end=" ", flush=True)
    cal_dfs = []
    for d in AGG_DATES[:7]:
        df = fetch_agg_daily(sym, d, SESSION)
        if df is not None and len(df) > 100:
            cal_dfs.append(df)
        if len(cal_dfs) >= 3:
            break
    if not cal_dfs:
        print("no data, skipping")
        continue

    sample = pd.concat(cal_dfs, ignore_index=True)
    daily_trades = len(sample) / len(cal_dfs)
    daily_vol = sample["quantity"].sum() / len(cal_dfs)
    tick_size = max(int(daily_trades / TARGET_BARS_DAY), 1)
    vol_size = max(daily_vol / TARGET_BARS_DAY, sample["quantity"].min())
    print(f"tick_size={tick_size:,} vol_size={vol_size:,.2f} (from {len(cal_dfs)} days)")
    del sample, cal_dfs

    # 2. Download + process in batches
    all_tick, all_vol = [], []
    n_batches = (len(AGG_DATES) + BATCH - 1) // BATCH
    trades_total = 0

    for bi in tqdm(range(n_batches), desc=f"  {sym}", leave=True):
        batch_dates = AGG_DATES[bi*BATCH : (bi+1)*BATCH]

        # Parallel download this batch
        batch_dfs = {}
        with ThreadPoolExecutor(max_workers=DL_WORKERS) as pool:
            futs = {pool.submit(fetch_agg_daily, sym, d, SESSION): d for d in batch_dates}
            for fut in as_completed(futs):
                d = futs[fut]
                r = fut.result()
                if r is not None and len(r) >= 10:
                    batch_dfs[d] = r

        # Process each day's trades into bars
        for d in sorted(batch_dfs.keys()):
            trades = batch_dfs[d]
            trades_total += len(trades)
            all_tick.append(build_tick_bars(trades, tick_size))
            all_vol.append(build_volume_bars(trades, vol_size))
        del batch_dfs

    # 3. Save
    nt, nv = 0, 0
    if all_tick:
        df = pd.concat(all_tick, ignore_index=True)
        save_parquet(df, f"{OUTPUT_DIR}/{sym}/tick_bar.parquet")
        nt = len(df)
        del df
    del all_tick
    if all_vol:
        df = pd.concat(all_vol, ignore_index=True)
        save_parquet(df, f"{OUTPUT_DIR}/{sym}/volume_bar.parquet")
        nv = len(df)
        del df
    del all_vol
    gc.collect()

    mem = psutil.virtual_memory()
    print(f"  Done: {trades_total:,} trades -> tick={nt:,} vol={nv:,} | RAM: {mem.available/1e9:.1f}GB free\n")

print(f"\nAll done! {len(todo)} symbols processed, {already} skipped")


## 7. Verification

In [ ]:
from pathlib import Path
import pandas as pd

base = Path(OUTPUT_DIR)
sym_dirs = sorted([d.name for d in base.iterdir() if d.is_dir()])

data_types = ["kline_1m", "kline_5m", "kline_15m", "kline_1h",
              "funding_rate", "metrics", "tick_bar", "volume_bar"]

report = []
for sym in sym_dirs:
    row = {"symbol": sym}
    for dt in data_types:
        p = base / sym / f"{dt}.parquet"
        if p.exists():
            df = pd.read_parquet(p, columns=["timestamp"])
            row[dt] = f"{len(df):,} ({df['timestamp'].min().date()}~{df['timestamp'].max().date()})"
        else:
            row[dt] = "MISSING"
    report.append(row)

rdf = pd.DataFrame(report).set_index("symbol")

# Summary
print(f"Symbols: {len(sym_dirs)}")
print(f"\nCompleteness:")
for dt in data_types:
    n = (rdf[dt] != "MISSING").sum()
    print(f"  {dt:<16} {n}/{len(sym_dirs)}")

# Disk usage
total = sum(f.stat().st_size for f in base.rglob("*.parquet"))
print(f"\nTotal disk: {total / 1e9:.1f} GB")

# Show any missing
missing = rdf[rdf.apply(lambda r: "MISSING" in r.values, axis=1)]
if not missing.empty:
    print(f"\nSymbols with missing data ({len(missing)}):")
    print(missing)

## 8. Copy to Local (optional)

After download completes, you can copy from Drive to your local machine:

```bash
# Option 1: rclone (fast)
rclone copy gdrive:ultraTM/data/merged data/merged -P

# Option 2: zip and download
# Run the cell below to create a zip, then download from Colab file browser
```

In [ ]:
# Uncomment to create zip for download
# !cd /content/drive/MyDrive/ultraTM/data && zip -r /content/merged.zip merged/ -x '*.DS_Store'